# Reference 3 — Data Quality and Tests

A notebook of **six fully worked reference answers**. Read the reasoning, run the code, test alternate inputs, and then try to solve the same task without looking.

**Level:** Intermediate · **Mode:** Worked reference solutions · **Prerequisite:** Complete related numbered lessons first.

**Instructions:** Write clear functions or short programs, not just expressions. State assumptions, test at least one ordinary case and one edge case, and avoid external network calls. Some exercises have multiple valid designs.

## 01. Typed and validated product

Static hints document a record, while runtime checks defend the boundary.

**Acceptance criteria:** Type hints and explicit validation are separate.

**Hint:** bool is subclass of int and may need exclusion.

In [ ]:
from typing import TypedDict

class Product(TypedDict):
    sku: str
    price: float

def parse_product(data: dict) -> Product:
    if not isinstance(data, dict) or not isinstance(data.get("sku"), str):
        raise ValueError("invalid SKU")
    price = data.get("price")
    if isinstance(price, bool) or not isinstance(price, (int, float)) or price < 0:
        raise ValueError("invalid price")
    return {"sku": data["sku"], "price": float(price)}

assert parse_product({"sku": "P1", "price": 5}) == {"sku": "P1", "price": 5.0}

## 02. Regex with date validation

Regex checks shape; datetime.date checks calendar validity.

**Acceptance criteria:** Leap-year boundaries checked.

**Hint:** A syntactically correct date might not exist.

In [ ]:
import re
from datetime import date
def parse_iso_date(text):
    match = re.fullmatch(r"(?P<year>\d{4})-(?P<month>\d{2})-(?P<day>\d{2})", text)
    if not match:
        raise ValueError("invalid date format")
    return date(*(int(match.group(key)) for key in ("year", "month", "day")))

assert parse_iso_date("2024-02-29") == date(2024, 2, 29)
try:
    parse_iso_date("2023-02-29")
except ValueError:
    print("Nonexistent date rejected")

## 03. CSV validation pipeline

Report row numbers on invalid input rather than stopping without context.

**Acceptance criteria:** Invalid row reported with source line number.

**Hint:** Keep independent lists of successes and errors.

In [ ]:
import csv
from io import StringIO
raw = "sku,quantity\nP1,3\nP2,broken\nP3,5\n"
valid, errors = [], []
for line, row in enumerate(csv.DictReader(StringIO(raw)), start=2):
    try:
        quantity = int(row["quantity"])
        if quantity <= 0:
            raise ValueError("quantity must be positive")
        valid.append((row["sku"], quantity))
    except (ValueError, KeyError, TypeError) as exc:
        errors.append((line, str(exc)))
assert valid == [("P1", 3), ("P3", 5)]
assert len(errors) == 1
print(valid, errors)

## 04. Exception chaining

Turn a low-level conversion exception into a domain-specific error without losing context.

**Acceptance criteria:** Original error remains available through __cause__.

**Hint:** Use raise ... from original_error.

In [ ]:
class QuantityError(ValueError):
    pass

def parse_quantity(text):
    try:
        return int(text)
    except ValueError as error:
        raise QuantityError(f"invalid quantity {text!r}") from error

try:
    parse_quantity("N/A")
except QuantityError as exc:
    assert isinstance(exc.__cause__, ValueError)

## 05. Unit tests and fixtures

Create fresh state for each test method and run the suite without Jupyter command-line interference.

**Acceptance criteria:** Tests can run independently.

**Hint:** Never reuse a global Cart instance between test methods.

In [ ]:
import unittest
class Cart:
    def __init__(self):
        self.items = []
    def add(self, value):
        self.items.append(value)

class CartTests(unittest.TestCase):
    def setUp(self):
        self.cart = Cart()
    def test_new_is_empty(self):
        self.assertEqual(self.cart.items, [])
    def test_add(self):
        self.cart.add("book")
        self.assertEqual(self.cart.items, ["book"])

suite = unittest.defaultTestLoader.loadTestsFromTestCase(CartTests)
assert unittest.TextTestRunner(verbosity=0).run(suite).wasSuccessful()

## 06. Mocking a collaborator

A fake sender avoids actual email or HTTP requests while verifying the interaction.

**Acceptance criteria:** Zero real network calls.

**Hint:** Mock the boundary rather than the function under test.

In [ ]:
from unittest.mock import Mock

def publish(sender, message):
    return sender.send(message)

sender = Mock()
sender.send.return_value = "ok"
assert publish(sender, "ready") == "ok"
sender.send.assert_called_once_with("ready")

## Reflection

- Which solution uses a reusable function rather than hidden notebook state?
- Have you tested normal, boundary, and invalid cases?
- What is the expected complexity or important design trade-off?
- Can you restart the kernel and run all cells cleanly?

**Reference notebooks:** See `../solutions/` for a selection of worked examples. The solutions deliberately do not cover every challenge; explain and test the remaining problems independently.